# Notebook 58 — Lindblad → Γ_eff(x) → b Pipeline


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit


In [ ]:
x = np.linspace(1e-3,1,400)
def gamma_eff(x, kind='baseline'):
    if kind=='baseline': return 2.0*np.ones_like(x)
    if kind=='dephasing': return 2.0 + 0.8*np.exp(-((x-0.3)/0.15)**2)
    if kind=='interaction': return 2.0*(1+0.3*np.sin(2*np.pi*x))
def build_S(gamma):
    dx=x[1]-x[0]
    integral=np.cumsum(gamma)*dx
    return np.exp(-integral)
cases=['baseline','dephasing','interaction']
S_data={c:build_S(gamma_eff(x,c)) for c in cases}


In [ ]:
def stretched(x,a,b): return np.exp(-a*x**b)
def fit(S): return curve_fit(stretched,x,S,p0=[1,1])[0]
fits={c:fit(S_data[c]) for c in cases}


In [ ]:
def extract_gamma(S): return -np.gradient(S,x)/np.clip(S,1e-12,None)
gamma_extracted={c:extract_gamma(S_data[c]) for c in cases}


In [ ]:
def b_local(gamma,S):
    integral=-np.log(np.clip(S,1e-12,None))
    return x*gamma/np.clip(integral,1e-12,None)
b_locals={c:b_local(gamma_extracted[c],S_data[c]) for c in cases}


In [ ]:
plt.figure();
for c in cases: plt.plot(x,S_data[c],label=f'{c}, b={fits[c][1]:.2f}')
plt.legend(); plt.title('Decay curves'); plt.show()

plt.figure();
for c in cases: plt.plot(x,gamma_extracted[c],label=c)
plt.legend(); plt.title('Γ_eff(x)'); plt.show()

plt.figure();
for c in cases: plt.plot(x,b_locals[c],label=c)
plt.legend(); plt.title('b_local(x)'); plt.show()
